In [10]:
!pip install librosa scikit-learn soundfile numpy

In [11]:
import os
import numpy as np
import librosa

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score


In [12]:
def extract_features(audio_path, sr=22050):

    signal, sr = librosa.load(audio_path, sr=sr)

    # =====================
    # MFCC (13)
    # =====================
    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=13
    )
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)
    mfcc_min = np.min(mfcc, axis=1)
    mfcc_max = np.max(mfcc, axis=1)

    # =====================
    # PLP aproximado manual
    # =====================

    # Espectrograma Mel
    mel_spec = librosa.feature.melspectrogram(
        y=signal,
        sr=sr,
        n_mels=26,
        power=2
    )

    # Compresión log
    log_mel = np.log(mel_spec + 1e-9)

    # DCT (similar a PLP)
    plp = librosa.feature.mfcc(
        S=log_mel,
        n_mfcc=13
    )

    plp_mean = np.mean(plp, axis=1)
    plp_std = np.std(plp, axis=1)
    plp_min = np.min(plp, axis=1)
    plp_max = np.max(plp, axis=1)

    # Concatenar
    features = np.concatenate([
    mfcc_mean, mfcc_std, mfcc_min, mfcc_max,
    plp_mean, plp_std, plp_min, plp_max
    ])

    return features


In [13]:
import os
import numpy as np

emotion_map = {
    "ALEGRIA": "positivo",
    "NEUTRO": "neutral",
    "TRISTEZA": "negativo",
    "MIEDO": "negativo",
    "ASCO": "negativo",
    "SORPRESA": "negativo",
    "IRA": "negativo"
}

emotion2id = {
    "positivo": 0,
    "neutral": 1,
    "negativo": 2
}

def load_dataset(dataset_path):
    X, y = [], []

    for file in os.listdir(dataset_path):

        if not file.lower().endswith(".wav"):
            continue

        name = os.path.splitext(file)[0]
        parts = name.split("_")

        if len(parts) < 3:
            print("Formato inesperado:", file)
            continue

        # La emoción está en la posición 1
        emotion_raw = parts[1].strip().upper()

        if emotion_raw not in emotion_map:
            print("Emoción no reconocida:", emotion_raw)
            continue

        label = emotion2id[emotion_map[emotion_raw]]

        filepath = os.path.join(dataset_path, file)
        features = extract_features(filepath)

        if features is None:
            continue

        X.append(features)
        y.append(label)

    print("Total muestras cargadas:", len(X))

    return np.array(X), np.array(y)

In [14]:
DATASET_PATH = "/content/dataset"

X, y = load_dataset(DATASET_PATH)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("===== MÉTRICAS GLOBALES =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print(classification_report(
    y_test,
    y_pred,
    target_names=["positivo", "neutral", "negativo"]
))


Total muestras cargadas: 2005
===== MÉTRICAS GLOBALES =====
Accuracy : 0.4938
Precision: 0.5328
Recall   : 0.6502
F1-score : 0.4948
              precision    recall  f1-score   support

    positivo       0.28      0.82      0.42        51
     neutral       0.49      0.82      0.62        95
    negativo       0.82      0.31      0.45       255

    accuracy                           0.49       401
   macro avg       0.53      0.65      0.49       401
weighted avg       0.68      0.49      0.48       401



In [15]:
#Carga del dataset
DATASET_PATH = "/content/dataset"

X, y = load_dataset(DATASET_PATH)

print("Muestras totales:", X.shape)

Total muestras cargadas: 2005
Muestras totales: (2005, 104)


In [16]:
#Division de train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [17]:
#Modelo SVM
model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced"
)

model.fit(X_train, y_train)

SVC(C=10, class_weight='balanced')

In [18]:
#Evaliacion
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nReporte de clasificación:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=["positivo", "neutral", "negativo"]
))


Accuracy: 0.4937655860349127

Reporte de clasificación:

              precision    recall  f1-score   support

    positivo       0.28      0.82      0.42        51
     neutral       0.49      0.82      0.62        95
    negativo       0.82      0.31      0.45       255

    accuracy                           0.49       401
   macro avg       0.53      0.65      0.49       401
weighted avg       0.68      0.49      0.48       401

